# Build sum-of-product vibrational KEO for formaldehyde ($\mathrm{H_2CO}$) using valence coordinates

In [1]:
import jax
import numpy as np
from jax import numpy as jnp
from scipy import constants

from gvib import Gvib
from keo_jax import batch_Gmat, com
from revad import dihedral, dot, norm

jax.config.update("jax_enable_x64", True)

G_to_invcm = (
    constants.value("Planck constant")
    * constants.value("Avogadro constant")
    * 1e16
    / (4.0 * np.pi**2 * constants.value("speed of light in vacuum"))
    * 1e5
)

Define the internal-to-Cartesian and Cartesian-to-internal coordinate transformations for $\mathrm{H_2CO}$.

The internal coordinates are defined as:

1. $r_{\mathrm{CO}}$ — C–O bond length,
2. $r_{\mathrm{CH_1}}$ — C–H1 bond length,
3. $r_{\mathrm{CH_2}}$ — C–H2 bond length,
4. $\alpha_{\mathrm{H_1CO}}$ — H1–C–O bond angle,
5. $\alpha_{\mathrm{H_2CO}}$ — H2–C–O bond angle,
6. $\Delta\tau = \tau + \pi$, where $\tau$ is the book angle (dihedral angle) between the (H1CO) and (H2CO) planes.

The atom ordering in Cartesian coordinates is:

$$
\mathrm{C},\ \mathrm{O},\ \mathrm{H_1},\ \mathrm{H_2}
$$

In [2]:
masses = np.array([12.0, 15.99491463, 1.00782505, 1.00782505])  # masses of C, O, H, H

ncoo = 6


def internal_to_cartesian(q):
    rCO, rCH1, rCH2, aOCH1, aOCH2, dtau = q
    tau = np.pi + dtau
    xyz = [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, rCO],
        [
            rCH1 * np.sin(aOCH1) * np.cos(tau * 0.5),
            -rCH1 * np.sin(aOCH1) * np.sin(tau * 0.5),
            rCH1 * np.cos(aOCH1),
        ],
        [
            rCH2 * np.sin(aOCH2) * np.cos(tau * 0.5),
            rCH2 * np.sin(aOCH2) * np.sin(tau * 0.5),
            rCH2 * np.cos(aOCH2),
        ],
    ]
    return np.array(xyz, dtype=object)


def cartesian_to_internal(xyz):
    C, O, H1, H2 = xyz

    CO = O - C
    CH1 = H1 - C
    CH2 = H2 - C

    rCO = norm(CO)
    rCH1 = norm(CH1)
    rCH2 = norm(CH2)

    aOCH1 = np.arccos(dot(CO, CH1) / (rCO * rCH1))
    aOCH2 = np.arccos(dot(CO, CH2) / (rCO * rCH2))

    tau = dihedral(H1, C, O, H2)
    dtau = tau + np.pi

    return np.array([rCO, rCH1, rCH2, aOCH1, aOCH2, dtau], dtype=object)

Define one-dimensional grids for the internal coordinate representation and construct the vibrational G-matrix in sum-of-products form

In [3]:
grids = [
    np.linspace(0.8, 1.2, 10),  # r_{CO}
    np.linspace(0.8, 1.2, 10),  # r_{CH1}
    np.linspace(0.8, 1.2, 10),  # r_{CH2}
    np.linspace(60 * np.pi / 180, 160 * np.pi / 180, 10),  # alpha_{H1CO}
    np.linspace(60 * np.pi / 180, 160 * np.pi / 180, 10),  # alpha_{H2CO}
    np.linspace(-60 * np.pi / 180, 60 * np.pi / 180, 10),  # dtau
]

gvib = Gvib(grids, masses, internal_to_cartesian, cartesian_to_internal)
gvib_sop = gvib.run()

print("Number of terms in Gvib_{i,j}\n i  j no.terms")
for i in range(ncoo):
    for j in range(i, ncoo):
        print(f"{i:2d} {j:2d} {gvib_sop[i, j].nterms:6d}")

Number of terms in Gvib_{i,j}
 i  j no.terms
 0  0      1
 0  1      1
 0  2      1
 0  3      2
 0  4      2
 0  5      1
 1  1      1
 1  2      3
 1  3      3
 1  4      6
 1  5      2
 2  2      1
 2  3      6
 2  4      3
 2  5      2
 3  3      5
 3  4     12
 3  5      4
 4  4      5
 4  5      4
 5  5     14


Evaluate the vibrational G-matrix numerically on the multidimensional grid formed by the Cartesian product of the one-dimensional grids used to construct the sum-of-products representation

In [4]:
def internal_to_cartesian_jax(q):
    rCO, rCH1, rCH2, aOCH1, aOCH2, dtau = q
    tau = jnp.pi + dtau
    xyz = [
        [0.0, 0.0, 0.0],
        [0.0, 0.0, rCO],
        [
            rCH1 * jnp.sin(aOCH1) * jnp.cos(tau * 0.5),
            -rCH1 * jnp.sin(aOCH1) * jnp.sin(tau * 0.5),
            rCH1 * jnp.cos(aOCH1),
        ],
        [
            rCH2 * jnp.sin(aOCH2) * jnp.cos(tau * 0.5),
            rCH2 * jnp.sin(aOCH2) * jnp.sin(tau * 0.5),
            rCH2 * jnp.cos(aOCH2),
        ],
    ]
    return jnp.array(xyz, dtype=jnp.float64)


grid_ndim = np.stack(np.meshgrid(*grids, indexing="ij"), axis=-1)
grid_ndim = grid_ndim.reshape(-1, len(grids))

gmat = batch_Gmat(grid_ndim, masses, internal_to_cartesian_jax)

gvib_ndim = gmat[:, :ncoo, :ncoo]

print("gvib_ndmm.shape = ", gvib_ndim.shape)

gvib_ndmm.shape =  (1000000, 6, 6)


Calculate point-wise differences between the sum-of-products and numerical vibrational G-matrix

In [5]:
# number of multidimensional grid points
ngrid = grid_ndim.shape[0]

# evaluate the sum-of-products G-matrix on the full direct-product grid
gvib_sop_eval = np.zeros((ngrid, ncoo, ncoo), dtype=np.float64)

# map each multidimensional grid point to its 1D grid indices
grid_indices = np.indices([len(grid) for grid in grids]).reshape(len(grids), -1).T

for ipoint, inds in enumerate(grid_indices):
    for i in range(ncoo):
        for j in range(ncoo):
            sop = gvib_sop[i, j]

            value = 0.0

            for iterm in range(sop.nterms):
                term = 1.0

                for k in range(ncoo):
                    term *= sop.terms[iterm][k][inds[k]]

                value += term

            gvib_sop_eval[ipoint, i, j] = value * G_to_invcm

In [6]:
# differences

diff = gvib_sop_eval - gvib_ndim
abs_diff = np.abs(diff)

print("Maximum absolute difference:", np.max(abs_diff))
print("\nMean absolute difference:", np.mean(abs_diff))
print("\nRMS difference:", np.sqrt(np.mean(diff**2)))

Maximum absolute difference: 3.637978807091713e-12

Mean absolute difference: 7.737507681715876e-15

RMS difference: 3.984065535686355e-14


In [8]:
print("Element-wise errors")

for i in range(ncoo):
    for j in range(i, ncoo):
        d = diff[:, i, j]
        max_abs = np.max(np.abs(d))
        rms = np.sqrt(np.mean(d**2))

        print(
            f"G[{i},{j}]  "
            f"max abs = {max_abs:.3e}   "
            f"RMS = {rms:.3e}   "
        )

Element-wise errors
G[0,0]  max abs = 3.553e-15   RMS = 1.514e-15   
G[0,1]  max abs = 3.553e-15   RMS = 5.688e-16   
G[0,2]  max abs = 4.996e-15   RMS = 5.765e-16   
G[0,3]  max abs = 1.221e-14   RMS = 9.939e-16   
G[0,4]  max abs = 7.550e-15   RMS = 9.671e-16   
G[0,5]  max abs = 1.185e-14   RMS = 2.861e-16   
G[1,1]  max abs = 4.974e-14   RMS = 1.566e-14   
G[1,2]  max abs = 5.773e-15   RMS = 9.872e-16   
G[1,3]  max abs = 3.797e-14   RMS = 6.044e-15   
G[1,4]  max abs = 1.954e-14   RMS = 2.106e-15   
G[1,5]  max abs = 6.040e-14   RMS = 3.579e-15   
G[2,2]  max abs = 4.974e-14   RMS = 1.568e-14   
G[2,3]  max abs = 1.688e-14   RMS = 2.115e-15   
G[2,4]  max abs = 3.220e-14   RMS = 6.046e-15   
G[2,5]  max abs = 8.527e-14   RMS = 3.535e-15   
G[3,3]  max abs = 3.837e-13   RMS = 3.042e-14   
G[3,4]  max abs = 1.741e-13   RMS = 8.681e-15   
G[3,5]  max abs = 4.619e-13   RMS = 9.033e-15   
G[4,4]  max abs = 3.837e-13   RMS = 3.060e-14   
G[4,5]  max abs = 5.116e-13   RMS = 9.624e-15   
